# Lab 4 – Neural Network Exercises

**Group 3**
- Brian Ezeanya (26134667)
- Joseph Gimba (25243888)
- Gloria Ezeanya (26128756)

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf

## Exercise 1

Weights from Figure 1 (no bias): x1→z11 = -1, x2→z11 = 0.5, x2→z12 = 0.5, z11→out = 1, z12→out = -1.
x1 is not connected to z12, so we keep that weight at 0.

**Forward pass with x = (1, 2)**

- z11 = (-1)(1) + (0.5)(2) = 0, so a11 = σ(0) = 0.5
- z12 = (0.5)(2) = 1, so a12 = σ(1) = 0.7311
- z21 = (1)(0.5) + (-1)(0.7311) = -0.2311, so ŷ = σ(-0.2311) = 0.4425

ŷ = 0.4425 is below 0.5, so the network predicts the wrong class since y = 1.

In [2]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x = np.array([1.0, 2.0])
y = 1
lr = 1

W1 = np.array([[-1.0, 0.5],
               [ 0.0, 0.5]])
w2 = np.array([1.0, -1.0])

# forward pass
a1 = sigmoid(W1 @ x)
y_hat = sigmoid(w2 @ a1)
print("a11 =", round(a1[0], 4), " a12 =", round(a1[1], 4))
print("y_hat =", round(y_hat, 4))

a11 = 0.5  a12 = 0.7311
y_hat = 0.4425


**Backward pass**

With sigmoid output and the log-likelihood, the error at the output is just y - ŷ. We send it back
through the old output weights, multiplied by the sigmoid derivative a(1 - a).
Since we maximise the log-likelihood, each weight is updated as: new w = old w + η × error × input.

- output error: 1 - 0.4425 = 0.5575
- error at z11: 0.5575 × (1) × 0.5 × 0.5 = 0.1394
- error at z12: 0.5575 × (-1) × 0.7311 × 0.2689 = -0.1096

New weights:

- z11→out: 1 + 0.5575 × 0.5 = 1.2788
- z12→out: -1 + 0.5575 × 0.7311 = -0.5924
- x1→z11: -1 + 0.1394 × 1 = -0.8606
- x2→z11: 0.5 + 0.1394 × 2 = 0.7788
- x2→z12: 0.5 + (-0.1096) × 2 = 0.2808

In [3]:
# backward pass
d2 = y - y_hat
d1 = d2 * w2 * a1 * (1 - a1)

w2_new = w2 + lr * d2 * a1
W1_new = W1 + lr * np.outer(d1, x)
W1_new[1, 0] = 0   # x1 -> z12 does not exist

print("output error:", round(d2, 4))
print("error z11:", round(d1[0], 4), " error z12:", round(d1[1], 4))
print()
print("z11 -> out:", round(w2_new[0], 4))
print("z12 -> out:", round(w2_new[1], 4))
print("x1 -> z11: ", round(W1_new[0, 0], 4))
print("x2 -> z11: ", round(W1_new[0, 1], 4))
print("x2 -> z12: ", round(W1_new[1, 1], 4))

output error: 0.5575
error z11: 0.1394  error z12: -0.1096

z11 -> out: 1.2788
z12 -> out: -0.5924
x1 -> z11:  -0.8606
x2 -> z11:  0.7788
x2 -> z12:  0.2808


**Second forward pass with the new weights**

- z11 = (-0.8606)(1) + (0.7788)(2) = 0.6969, so a11 = 0.6675
- z12 = (0.2808)(2) = 0.5615, so a12 = 0.6368
- z21 = (1.2788)(0.6675) + (-0.5924)(0.6368) = 0.4763, so ŷ = 0.6169

In [4]:
# forward pass again with new weights
a1_new = sigmoid(W1_new @ x)
y_hat_new = sigmoid(w2_new @ a1_new)
print("y_hat before:", round(y_hat, 4))
print("y_hat after: ", round(y_hat_new, 4))

y_hat before: 0.4425
y_hat after:  0.6169


After one update ŷ went from 0.44 to 0.62, so now it predicts the correct class. The weights on the path
through z11 got bigger and the negative weight from z12 got smaller, which pushes the output up towards 1.
Because the learning rate is 1 the change is quite big for just one step.

## Exercise 2

XOR can't be done with one neuron, because a single neuron draws one straight line and the XOR points
can't be split by one line. So we need a hidden layer. We use XOR = OR and not AND.

Network: 2 inputs → 2 hidden neurons → 1 output neuron (3 neurons in total).
All neurons use a step activation: output 1 if input >= 0, else 0.

- h1 (OR): weights 1, 1 and bias -0.5
- h2 (AND): weights 1, 1 and bias -1.5
- output: weights 1 (from h1), -1 (from h2) and bias -0.5

In [5]:
def step(z):
    return (z >= 0).astype(int)

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

h1 = step(X[:, 0] + X[:, 1] - 0.5)
h2 = step(X[:, 0] + X[:, 1] - 1.5)
out = step(h1 - h2 - 0.5)

print("x1 x2 | h1 h2 | out")
for xi, a, b, o in zip(X, h1, h2, out):
    print(f" {xi[0]}  {xi[1]} |  {a}  {b} |  {o}")

x1 x2 | h1 h2 | out
 0  0 |  0  0 |  0
 0  1 |  1  0 |  1
 1  0 |  1  0 |  1
 1  1 |  1  1 |  0


## Exercise 3

In [6]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(10,)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax"),
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 339 (1.32 KB)

 Trainable params: 339 (1.32 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
X_train = tf.random.normal((100, 10))
y_train = tf.random.uniform((100,), minval=0, maxval=3, dtype=tf.int32)

history = model.fit(X_train, y_train, epochs=10, verbose=2)

Epoch 1/10


4/4 - 1s - 159ms/step - accuracy: 0.3700 - loss: 1.0841


Epoch 2/10


4/4 - 0s - 6ms/step - accuracy: 0.3700 - loss: 1.0770


Epoch 3/10


...

4/4 - 0s - 5ms/step - accuracy: 0.3900 - loss: 1.0559


Epoch 9/10


4/4 - 0s - 4ms/step - accuracy: 0.4000 - loss: 1.0532


Epoch 10/10


4/4 - 0s - 4ms/step - accuracy: 0.4100 - loss: 1.0506


The accuracy stays low (around 33-40%, where 33% is just guessing) because the labels are random, so there is nothing real for the model to learn.